# Entrega 2 - Perfilado, Diccionario y Limpieza Inicial (Consolidado)
## Data Visualization (1ACC0211) - Universidad Peruana de Ciencias Aplicadas

**Proyecto:** Dinamica del comercio mundial: patrones de exportacion e importacion por pais, categoria de producto y region geografica (1989-2023)

| Codigo | Nombre |
|---|---|
| U202218912 | Julio Cesar Meza Alfaro |
| U202212675 | Rosa Maria Rodriguez Valencia |
| U202214069 | Braulio Alonso Bartra Sandoval |

---

### Objetivo del notebook
Este notebook cubre el Entregable 2:
1. Perfilado completo del dataset (unidad de analisis, granularidad, tipos, nulos, cardinalidad)
2. Diccionario de datos
3. Identificacion de problemas de calidad
4. Reglas y ejecucion de limpieza
5. Dataset limpio exportado para Tableau
6. Bitacora de transformaciones
7. Modelado de datos

## 0. Configuracion e importaciones

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 80)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.width', 200)

RUTA_ORIGINAL = '../data/raw/34_years_world_export_import_dataset.csv'
RUTA_LIMPIO = '../data/processed/dataset_limpio_entrega2_consolidado.csv'
RUTA_AGREGADOS = '../data/processed/dataset_agregados_referencia.csv'
RUTA_PERFIL = '../data/processed/tabla_perfilado_entrega2_consolidado.csv'
RUTA_BITACORA = '../data/processed/bitacora_transformaciones_entrega2_consolidado.csv'

print('Librerias cargadas correctamente.')
print('Pandas {} | NumPy {}'.format(pd.__version__, np.__version__))

Librerias cargadas correctamente.
Pandas 2.3.3 | NumPy 2.2.6


## 1. Carga y exploracion inicial

In [2]:
df = pd.read_csv(RUTA_ORIGINAL)

print('=' * 60)
print('DIMENSIONES DEL DATASET')
print('=' * 60)
print('  Filas    : {:,}'.format(df.shape[0]))
print('  Columnas : {}'.format(df.shape[1]))
print('  Memoria  : {:.2f} MB'.format(df.memory_usage(deep=True).sum() / 1024**2))
print()
print('RANGO TEMPORAL')
print('  Anio minimo : {}'.format(df['Year'].min()))
print('  Anio maximo : {}'.format(df['Year'].max()))
print('  Anios unicos: {}'.format(df['Year'].nunique()))
print()
print('COBERTURA GEOGRAFICA')
print('  Paises / territorios unicos: {}'.format(df['Partner Name'].nunique()))
print()
print('PRIMERAS 5 FILAS')
df.head()

DIMENSIONES DEL DATASET
  Filas    : 8,096
  Columnas : 33
  Memoria  : 2.50 MB

RANGO TEMPORAL
  Anio minimo : 1988
  Anio maximo : 2021
  Anios unicos: 34

COBERTURA GEOGRAFICA
  Paises / territorios unicos: 265

PRIMERAS 5 FILAS


,Partner Name,Year,Export (US$ Thousand),Import (US$ Thousand),Export Product Share (%),Import Product Share (%),Revealed comparative advantage,World Growth (%),Country Growth (%),AHS Simple Average (%),AHS Weighted Average (%),AHS Total Tariff Lines,AHS Dutiable Tariff Lines Share (%),AHS Duty Free Tariff Lines Share (%),AHS Specific Tariff Lines Share (%),AHS AVE Tariff Lines Share (%),AHS MaxRate (%),AHS MinRate (%),AHS SpecificDuty Imports (US$ Thousand),AHS Dutiable Imports (US$ Thousand),AHS Duty Free Imports (US$ Thousand),MFN Simple Average (%),MFN Weighted Average (%),MFN Total Tariff Lines,MFN Dutiable Tariff Lines Share (%),MFN Duty Free Tariff Lines Share (%),MFN Specific Tariff Lines Share (%),MFN AVE Tariff Lines Share (%),MFN MaxRate (%),MFN MinRate (%),MFN SpecificDuty Imports (US$ Thousand),MFN Dutiable Imports (US$ Thousand),MFN Duty Free Imports (US$ Thousand)
0,Aruba,1988,"3,498.10",328.49,100.00,100,NaN,NaN,NaN,2.80,2.92,155.00,18.06,60.00,20.00,1.94,50.00,0.00,"1,867.00","2,346.37",781.65,13.59,8.46,"1,152.00",63.54,22.74,70.32,31.61,352.69,0.00,"2,186.00","3,128.02",0.00
1,Afghanistan,1988,"213,030.40","54,459.52",100.00,100,NaN,NaN,NaN,0.88,1.83,548.00,8.76,82.66,8.03,0.55,35.00,0.00,"30,863.03","70,204.13","23,987.37",17.68,12.43,"4,142.00",69.41,15.64,72.45,40.51,"2,029.66",0.00,"78,436.91","94,191.50",0.00
2,Angola,1988,"375,527.89","370,702.76",100.00,100,NaN,NaN,NaN,2.02,3.89,633.00,25.43,69.19,5.37,0.00,40.00,0.00,"723,819.51","754,183.84","167,297.68",12.70,6.14,"5,438.00",76.00,16.27,41.55,24.80,451.15,0.00,"727,741.99","921,481.52",0.00
3,Anguila,1988,366.98,4.00,100.00,100,NaN,NaN,NaN,3.71,1.09,33.00,6.06,72.73,21.21,0.00,35.00,0.00,60.00,65.00,518.00,16.63,14.75,322.00,66.15,22.05,78.79,36.36,100.00,0.00,94.00,583.00,0.00
4,Albania,1988,"30,103.56","47,709.30",100.00,100,NaN,NaN,NaN,1.84,2.38,744.00,20.83,60.48,17.61,1.08,25.00,0.00,"18,806.15","62,294.53","38,901.42",19.20,9.68,"5,684.00",66.87,19.19,57.93,48.52,"3,000.00",0.00,"37,904.09","101,195.95",0.00


**Observacion inicial:** el dataset no es solo grande, tambien es heterogeneo. Hay variables comerciales, arancelarias y de contexto historico que no se comportan igual entre si.
La cobertura temporal llega hasta 2021 y la cobertura geografica mezcla paises vigentes con entidades historicas y agregados regionales. Por eso, antes de limpiar, primero hay que separar bien que es analitico y que es solo referencia historica.

In [3]:
print('TIPOS DE DATOS')
df.dtypes.to_frame('Tipo')

TIPOS DE DATOS


,Tipo
Partner Name,object
Year,int64
Export (US$ Thousand),float64
Import (US$ Thousand),float64
Export Product Share (%),float64
Import Product Share (%),int64
Revealed comparative advantage,float64
World Growth (%),float64
Country Growth (%),float64
AHS Simple Average (%),float64


## 2. Unidad de analisis y granularidad

In [4]:
n_paises = df['Partner Name'].nunique()
n_anios = df['Year'].nunique()
anio_min = df['Year'].min()
anio_max = df['Year'].max()
duplicados = df.duplicated(subset=['Partner Name', 'Year']).sum()
esperado = n_paises * n_anios

print('=' * 60)
print('UNIDAD DE ANALISIS Y GRANULARIDAD')
print('=' * 60)
print('  Unidad de analisis : Pais x Anio')
print('  Granularidad       : Anual')
print('  Paises/territorios : {}'.format(n_paises))
print('  Anios cubiertos    : {} ({}-{})'.format(n_anios, anio_min, anio_max))
print('  Registros totales  : {:,}'.format(len(df)))
print('  Esperado (paises x anios): {:,}'.format(esperado))
print('  Duplicados (Pais x Anio): {}'.format(duplicados))

UNIDAD DE ANALISIS Y GRANULARIDAD
  Unidad de analisis : Pais x Anio
  Granularidad       : Anual
  Paises/territorios : 265
  Anios cubiertos    : 34 (1988-2021)
  Registros totales  : 8,096
  Esperado (paises x anios): 9,010
  Duplicados (Pais x Anio): 0


**Conclusión de granularidad:** la unidad de análisis correcta es `pais x anio`, pero eso no significa que todas las combinaciones posibles deban existir. En este tipo de serie histórica es normal encontrar huecos porque algunos territorios aparecen, cambian de nombre o dejan de existir durante el periodo.
Por eso, el objetivo de la limpieza no es forzar una cuadrícula perfecta, sino conservar el histórico real y separar las excepciones que no deben entrar al análisis principal.

## 3. Diccionario de datos

In [5]:
diccionario = pd.DataFrame([
    ['Partner Name', 'str', 'Dimension', 'Pais o territorio que reporta el flujo comercial', 'Incluye agregados regionales. Se limpia y filtra.'],
    ['Year', 'int64', 'Temporal', 'Anio calendario del flujo comercial', 'Rango 1988-2021. Clave primaria con Partner Name.'],
    ['Export (US$ Thousand)', 'float64', 'Metrica', 'Valor total de exportaciones en miles de USD corrientes', 'Sin ajuste por inflacion. 20 valores en cero.'],
    ['Import (US$ Thousand)', 'float64', 'Metrica', 'Valor total de importaciones en miles de USD corrientes', 'Sin ajuste por inflacion. Sin valores en cero.'],
    ['Export Product Share (%)', 'float64', 'Constante', 'Participacion del producto en exportaciones del pais', 'Siempre 100. Columna sin valor -> eliminar.'],
    ['Import Product Share (%)', 'int64', 'Constante', 'Participacion del producto en importaciones del pais', 'Siempre 100. Columna sin valor -> eliminar.'],
    ['Revealed comparative advantage', 'float64', 'Constante', 'Indice de ventaja comparativa revelada', 'Siempre 1.0 cuando no nulo. Columna sin valor -> eliminar.'],
    ['World Growth (%)', 'float64', 'Metrica', 'Crecimiento del comercio mundial respecto al anio anterior', 'Se completa por anio cuando hay valor.'],
    ['Country Growth (%)', 'float64', 'Redundante', 'Crecimiento del comercio del pais', 'Redundante con World Growth. Columna sin valor -> eliminar.'],
    ['AHS Simple Average (%)', 'float64', 'Metrica', 'Promedio simple de aranceles AHS (%)', '0.2% nulos.'],
    ['AHS Weighted Average (%)', 'float64', 'Metrica', 'Promedio ponderado de aranceles AHS (%)', '0.2% nulos. Max 197.76%.'],
    ['AHS Total Tariff Lines', 'float64', 'Metrica', 'Numero total de lineas arancelarias AHS', '0.2% nulos.'],
    ['AHS Dutiable Tariff Lines Share (%)', 'float64', 'Metrica', 'Porcentaje de lineas AHS con arancel positivo', '0.2% nulos.'],
    ['AHS Duty Free Tariff Lines Share (%)', 'float64', 'Metrica', 'Porcentaje de lineas AHS libres de arancel', '0.2% nulos.'],
    ['AHS Specific Tariff Lines Share (%)', 'float64', 'Metrica', 'Porcentaje de lineas AHS con arancel especifico', '0.2% nulos.'],
    ['AHS AVE Tariff Lines Share (%)', 'float64', 'Metrica', 'Porcentaje de lineas AHS con AVE', '0.2% nulos.'],
    ['AHS MaxRate (%)', 'float64', 'Metrica', 'Tasa arancelaria maxima AHS', 'Valores extremos; se conservan.'],
    ['AHS MinRate (%)', 'float64', 'Metrica', 'Tasa arancelaria minima AHS', 'Min 0.'],
    ['AHS SpecificDuty Imports (US$ Thousand)', 'float64', 'Metrica', 'Importaciones con arancel especifico AHS (miles USD)', '0.19% nulos.'],
    ['AHS Dutiable Imports (US$ Thousand)', 'float64', 'Metrica', 'Importaciones gravadas AHS (miles USD)', '0.19% nulos.'],
    ['AHS Duty Free Imports (US$ Thousand)', 'float64', 'Metrica', 'Importaciones libres AHS (miles USD)', '0.19% nulos.'],
    ['MFN Simple Average (%)', 'float64', 'Metrica', 'Promedio simple de aranceles MFN (%)', '0.19% nulos.'],
    ['MFN Weighted Average (%)', 'float64', 'Metrica', 'Promedio ponderado de aranceles MFN (%)', '0.19% nulos.'],
    ['MFN Total Tariff Lines', 'float64', 'Metrica', 'Numero total de lineas arancelarias MFN', '0.19% nulos.'],
    ['MFN Dutiable Tariff Lines Share (%)', 'float64', 'Metrica', 'Porcentaje de lineas MFN con arancel positivo', '0.19% nulos.'],
    ['MFN Duty Free Tariff Lines Share (%)', 'float64', 'Metrica', 'Porcentaje de lineas MFN libres de arancel', '0.19% nulos.'],
    ['MFN Specific Tariff Lines Share (%)', 'float64', 'Metrica', 'Porcentaje de lineas MFN con arancel especifico', '0.20% nulos.'],
    ['MFN AVE Tariff Lines Share (%)', 'float64', 'Metrica', 'Porcentaje de lineas MFN con AVE', 'Puede ser >100 (AVE). Se agrega flag.'],
    ['MFN MaxRate (%)', 'float64', 'Metrica', 'Tasa arancelaria maxima MFN', 'Valores extremos; se conservan.'],
    ['MFN MinRate (%)', 'float64', 'Constante', 'Tasa arancelaria minima MFN', 'Siempre 0. Columna sin valor -> eliminar.'],
    ['MFN SpecificDuty Imports (US$ Thousand)', 'float64', 'Metrica', 'Importaciones con arancel especifico MFN (miles USD)', '0.19% nulos.'],
    ['MFN Dutiable Imports (US$ Thousand)', 'float64', 'Metrica', 'Importaciones gravadas MFN (miles USD)', '0.19% nulos.'],
    ['MFN Duty Free Imports (US$ Thousand)', 'float64', 'Metrica', 'Importaciones libres MFN (miles USD)', '0.19% nulos.'],
], columns=['Variable', 'Tipo original', 'Rol analitico', 'Descripcion', 'Observaciones'])

print('Diccionario: {} variables documentadas'.format(len(diccionario)))
diccionario

Diccionario: 33 variables documentadas


,Variable,Tipo original,Rol analitico,Descripcion,Observaciones
0,Partner Name,str,Dimension,Pais o territorio que reporta el flujo comercial,Incluye agregados regionales. Se limpia y filtra.
1,Year,int64,Temporal,Anio calendario del flujo comercial,Rango 1988-2021. Clave primaria con Partner Name.
2,Export (US$ Thousand),float64,Metrica,Valor total de exportaciones en miles de USD c...,Sin ajuste por inflacion. 20 valores en cero.
3,Import (US$ Thousand),float64,Metrica,Valor total de importaciones en miles de USD c...,Sin ajuste por inflacion. Sin valores en cero.
4,Export Product Share (%),float64,Constante,Participacion del producto en exportaciones de...,Siempre 100. Columna sin valor -> eliminar.
5,Import Product Share (%),int64,Constante,Participacion del producto en importaciones de...,Siempre 100. Columna sin valor -> eliminar.
6,Revealed comparative advantage,float64,Constante,Indice de ventaja comparativa revelada,Siempre 1.0 cuando no nulo. Columna sin valor ...
7,World Growth (%),float64,Metrica,Crecimiento del comercio mundial respecto al a...,Se completa por anio cuando hay valor.
8,Country Growth (%),float64,Redundante,Crecimiento del comercio del pais,Redundante con World Growth. Columna sin valor...
9,AHS Simple Average (%),float64,Metrica,Promedio simple de aranceles AHS (%),0.2% nulos.


## 4. Perfilado completo

In [6]:
perfil_rows = []

for col in df.columns:
    nulos = df[col].isnull().sum()
    pct_nulos = round(nulos / len(df) * 100, 2)
    cardinalidad = df[col].nunique()

    if pd.api.types.is_numeric_dtype(df[col]):
        val_min = round(float(df[col].min()), 2)
        val_max = round(float(df[col].max()), 2)
        media = round(float(df[col].mean()), 2)
        desv_std = round(float(df[col].std()), 2)
        ceros = int((df[col] == 0).sum())
    else:
        val_min = 'N/A'
        val_max = 'N/A'
        media = 'N/A'
        desv_std = 'N/A'
        ceros = 'N/A'

    perfil_rows.append({
        'Variable': col,
        'Tipo': str(df[col].dtype),
        'Nulos': nulos,
        '% Nulos': pct_nulos,
        'Cardinalidad': cardinalidad,
        'Min': val_min,
        'Max': val_max,
        'Media': media,
        'Desv. Estandar': desv_std,
        'Valores en 0': ceros
    })

tabla_perfil = pd.DataFrame(perfil_rows)
print('Tabla de perfilado: {} variables analizadas'.format(len(tabla_perfil)))
tabla_perfil

tabla_perfil.to_csv(RUTA_PERFIL, index=False)

Tabla de perfilado: 33 variables analizadas


**Lectura del perfilado:** el dataset ya deja ver una regla de oro: no todo campo con muchos nulos es descartable, pero sí obliga a distinguir entre variables estructurales y variables de contexto.
Las columnas constantes o redundantes no aportan capacidad explicativa y, en cambio, pueden confundir el modelado posterior. En cambio, las variables de crecimiento, aranceles y comercio sí sostienen la narrativa del dashboard, aunque algunas necesiten imputación puntual o un tratamiento especial.

## 5. Identificacion de problemas de calidad

In [7]:
print('HALLAZGO 1 - Columnas constantes o redundantes')
constantes = {
    'Export Product Share (%)': df['Export Product Share (%)'].dropna().unique(),
    'Import Product Share (%)': df['Import Product Share (%)'].dropna().unique(),
    'Revealed comparative advantage': df['Revealed comparative advantage'].dropna().unique(),
    'MFN MinRate (%)': df['MFN MinRate (%)'].dropna().unique(),
}

for col, vals in constantes.items():
    print('  {} -> {} (constante)'.format(col, vals))

mask = df['World Growth (%)'].notna() & df['Country Growth (%)'].notna()
son_iguales = (df.loc[mask, 'World Growth (%)'] == df.loc[mask, 'Country Growth (%)']).all()
print('  Country Growth == World Growth en {} registros no nulos: {}'.format(mask.sum(), son_iguales))

cols_borrar = [
    'Export Product Share (%)',
    'Import Product Share (%)',
    'Revealed comparative advantage',
    'MFN MinRate (%)',
    'Country Growth (%)',
]

print('  Columnas a eliminar: {}'.format(cols_borrar))

HALLAZGO 1 - Columnas constantes o redundantes
  Export Product Share (%) -> [100.] (constante)
  Import Product Share (%) -> [100] (constante)
  Revealed comparative advantage -> [1.] (constante)
  MFN MinRate (%) -> [0.] (constante)
  Country Growth == World Growth en 4410 registros no nulos: True
  Columnas a eliminar: ['Export Product Share (%)', 'Import Product Share (%)', 'Revealed comparative advantage', 'MFN MinRate (%)', 'Country Growth (%)']


In [8]:
print('HALLAZGO 2 - Agregados regionales mezclados con paises')
regional_aggs = [
    'World', 'East Asia & Pacific', 'Europe & Central Asia',
    'Latin America & Caribbean', 'Middle East & North Africa',
    'North America', 'South Asia', 'Sub-Saharan Africa'
]
nombre_limpio = df['Partner Name'].str.strip()
agg_mask = nombre_limpio.isin(regional_aggs)
agg_counts = nombre_limpio[agg_mask].value_counts()
print(agg_counts.to_string())
print('Total filas agregadas: {}'.format(agg_mask.sum()))

HALLAZGO 2 - Agregados regionales mezclados con paises
Partner Name
World                         34
East Asia & Pacific           34
Europe & Central Asia         34
Latin America & Caribbean     34
Middle East & North Africa    34
North America                 34
South Asia                    34
Sub-Saharan Africa            34
Total filas agregadas: 272


In [9]:
print('HALLAZGO 3 - Exportaciones en cero')
zero_exp = df[df['Export (US$ Thousand)'] == 0]
print('Filas con exportaciones = 0: {}'.format(len(zero_exp)))
print(zero_exp[['Partner Name', 'Year']].head(10).to_string(index=False))

HALLAZGO 3 - Exportaciones en cero
Filas con exportaciones = 0: 20
  Partner Name  Year
Western Sahara  1988
Br. Antr. Terr  1991
  Neutral Zone  2000
  Us Msc.Pac.I  2000
Br. Antr. Terr  2001
  Neutral Zone  2001
        Monaco  2002
  Us Msc.Pac.I  2002
Br. Antr. Terr  2003
  Neutral Zone  2003


In [10]:
print('HALLAZGO 4 - MFN AVE > 100 (equivalente ad valorem)')
serie = df['MFN AVE Tariff Lines Share (%)']
no_nulos = serie.notna().sum()
over100 = (serie > 100).sum()
print('Registros no nulos : {:,}'.format(no_nulos))
print('Valores > 100     : {:,} ({:.1f}%)'.format(over100, (over100 / no_nulos * 100) if no_nulos > 0 else 0))
print('Maximo            : {:.2f}%'.format(serie.max()))
print('Mediana           : {:.2f}%'.format(serie.median()))

HALLAZGO 4 - MFN AVE > 100 (equivalente ad valorem)
Registros no nulos : 8,080
Valores > 100     : 3,641 (45.1%)
Maximo            : 3860.00%
Mediana           : 88.05%


In [11]:
print('HALLAZGO 5 - Nulos y outliers en variables arancelarias')
tariff_cols = [c for c in df.columns if c.startswith('AHS') or c.startswith('MFN')]
nulos_tarifas = df[tariff_cols].isnull().sum()
print(nulos_tarifas[nulos_tarifas > 0].to_string())
print('Total columnas arancelarias: {}'.format(len(tariff_cols)))
print()
print('AHS MaxRate max: {:.1f}%'.format(df['AHS MaxRate (%)'].max()))
print('MFN MaxRate max: {:.1f}%'.format(df['MFN MaxRate (%)'].max()))

HALLAZGO 5 - Nulos y outliers en variables arancelarias
AHS Simple Average (%)                     16
AHS Weighted Average (%)                   16
AHS Total Tariff Lines                     16
AHS Dutiable Tariff Lines Share (%)        16
AHS Duty Free Tariff Lines Share (%)       16
AHS Specific Tariff Lines Share (%)        16
AHS AVE Tariff Lines Share (%)             16
AHS MaxRate (%)                            16
AHS MinRate (%)                            16
AHS SpecificDuty Imports (US$ Thousand)    15
AHS Dutiable Imports (US$ Thousand)        15
AHS Duty Free Imports (US$ Thousand)       15
MFN Simple Average (%)                     15
MFN Weighted Average (%)                   15
MFN Total Tariff Lines                     15
MFN Dutiable Tariff Lines Share (%)        15
MFN Duty Free Tariff Lines Share (%)       15
MFN Specific Tariff Lines Share (%)        16
MFN AVE Tariff Lines Share (%)             16
MFN MaxRate (%)                            15
MFN MinRate (%)         

**Conclusiones de calidad:** los hallazgos no apuntan a errores aislados, sino a patrones claros: columnas constantes, agregados regionales mezclados con países, exportaciones codificadas en cero y una señal fuerte de que `World Growth (%)` debe reconstruirse por año.
La decisión más importante es conservar el histórico real sin contaminar el dataset analítico. Por eso, algunas filas se eliminan, otras se etiquetan y otras se transforman. Esa combinación es la que deja el notebook consistente para análisis y visualización.

## 6. Reglas de limpieza y transformaciones

In [12]:
df_clean = df.copy()
n_filas_inicial = len(df_clean)
print('Dataset base copiado: {} filas x {} columnas'.format(df_clean.shape[0], df_clean.shape[1]))

Dataset base copiado: 8096 filas x 33 columnas


In [13]:
# REGLA 1 - Eliminar columnas constantes o redundantes
COLS_BASURA = [
    'Export Product Share (%)',
    'Import Product Share (%)',
    'Revealed comparative advantage',
    'MFN MinRate (%)',
    'Country Growth (%)',
]

n_cols_antes = df_clean.shape[1]
df_clean = df_clean.drop(columns=COLS_BASURA)
n_cols_despues = df_clean.shape[1]

print('REGLA 1 - Columnas eliminadas: {}'.format(COLS_BASURA))
print('  Columnas antes: {} | despues: {}'.format(n_cols_antes, n_cols_despues))

REGLA 1 - Columnas eliminadas: ['Export Product Share (%)', 'Import Product Share (%)', 'Revealed comparative advantage', 'MFN MinRate (%)', 'Country Growth (%)']
  Columnas antes: 33 | despues: 28


In [14]:
# REGLA 2 - Limpiar espacios en Partner Name
antes_espacios = df_clean['Partner Name'].str.startswith(' ').sum()
df_clean['Partner Name'] = df_clean['Partner Name'].str.strip()
despues_espacios = df_clean['Partner Name'].str.startswith(' ').sum()
print('REGLA 2 - Espacios al inicio: {} -> {}'.format(antes_espacios, despues_espacios))

REGLA 2 - Espacios al inicio: 34 -> 0


In [15]:
# REGLA 3 - Separar agregados regionales
regional_aggs = [
    'World', 'East Asia & Pacific', 'Europe & Central Asia',
    'Latin America & Caribbean', 'Middle East & North Africa',
    'North America', 'South Asia', 'Sub-Saharan Africa'
]
mask_aggs = df_clean['Partner Name'].isin(regional_aggs)
df_agregados = df_clean[mask_aggs].copy()
n_agregados = len(df_agregados)
df_clean = df_clean[~mask_aggs].copy()
print('REGLA 3 - Filas agregadas separadas: {}'.format(n_agregados))
print('  Filas en dataset analitico: {:,}'.format(len(df_clean)))

REGLA 3 - Filas agregadas separadas: 272
  Filas en dataset analitico: 7,824


In [16]:
# REGLA 4 - Clasificar entidades historicas (entity_status)
PAISES_EXTINTOS = [
    'Soviet Union',
    'German Democratic Republic',
    'Yugoslavia,FR(Serbia/Montenegr',
    'Czechoslovakia',
    'Yemen Democratic',
    'Pacific Islands',
    'Ethiopia(includes Eritrea)',
]

df_clean['entity_status'] = np.where(
    df_clean['Partner Name'].isin(PAISES_EXTINTOS),
    'Extinto',
    'Activo'
)
n_extintos = int((df_clean['entity_status'] == 'Extinto').sum())
print('REGLA 4 - entity_status creado. Extintos: {}'.format(n_extintos))

REGLA 4 - entity_status creado. Extintos: 28


In [17]:
# REGLA 5 - Excluir entidades con cobertura insuficiente
cobertura = df_clean.groupby('Partner Name')['Year'].count()
excluir_cobertura = cobertura[(cobertura < 10) & (~cobertura.index.isin(PAISES_EXTINTOS))].index.tolist()
n_antes_cobertura = len(df_clean)
df_clean = df_clean[~df_clean['Partner Name'].isin(excluir_cobertura)].copy()
n_despues_cobertura = len(df_clean)
n_excluidos_cobertura = n_antes_cobertura - n_despues_cobertura
print('REGLA 5 - Territorios excluidos: {}'.format(excluir_cobertura))
print('  Filas eliminadas: {}'.format(n_excluidos_cobertura))

REGLA 5 - Territorios excluidos: ['French Guiana', 'Guadeloupe', 'Martinique', 'Reunion', 'Saint Barthélemy']
  Filas eliminadas: 41


In [18]:
# REGLA 6 - Completar World Growth por anio (broadcast)
nulos_antes_wg = int(df_clean['World Growth (%)'].isnull().sum())
world_growth_by_year = df_clean.groupby('Year')['World Growth (%)'].first()
df_clean['World Growth (%)'] = df_clean['Year'].map(world_growth_by_year)
nulos_despues_wg = int(df_clean['World Growth (%)'].isnull().sum())
n_rellenos_wg = nulos_antes_wg - nulos_despues_wg
print('REGLA 6 - World Growth completado. Nulos: {} -> {} (rellenos: {})'.format(nulos_antes_wg, nulos_despues_wg, n_rellenos_wg))

REGLA 6 - World Growth completado. Nulos: 3410 -> 195 (rellenos: 3215)


In [19]:
# REGLA 7 - Imputar nulos menores en variables AHS/MFN
tariff_cols = [c for c in df_clean.columns if c.startswith('AHS') or c.startswith('MFN')]
imputaciones = {}
for col in tariff_cols:
    n_null = int(df_clean[col].isnull().sum())
    if n_null > 0:
        df_clean[col] = df_clean[col].fillna(df_clean.groupby('Year')[col].transform('median'))
        imputaciones[col] = n_null

total_imputados_tarifas = sum(imputaciones.values())
print('REGLA 7 - Nulos imputados en tarifas: {}'.format(total_imputados_tarifas))
if imputaciones:
    print('  Columnas afectadas:')
    for col, n_null in imputaciones.items():
        print('    {} -> {} nulos imputados'.format(col, n_null))

REGLA 7 - Nulos imputados en tarifas: 356
  Columnas afectadas:
    AHS Simple Average (%) -> 16 nulos imputados
    AHS Weighted Average (%) -> 16 nulos imputados
    AHS Total Tariff Lines -> 16 nulos imputados
    AHS Dutiable Tariff Lines Share (%) -> 16 nulos imputados
    AHS Duty Free Tariff Lines Share (%) -> 16 nulos imputados
    AHS Specific Tariff Lines Share (%) -> 16 nulos imputados
    AHS AVE Tariff Lines Share (%) -> 16 nulos imputados
    AHS MaxRate (%) -> 16 nulos imputados
    AHS MinRate (%) -> 16 nulos imputados
    AHS SpecificDuty Imports (US$ Thousand) -> 15 nulos imputados
    AHS Dutiable Imports (US$ Thousand) -> 15 nulos imputados
    AHS Duty Free Imports (US$ Thousand) -> 15 nulos imputados
    MFN Simple Average (%) -> 15 nulos imputados
    MFN Weighted Average (%) -> 15 nulos imputados
    MFN Total Tariff Lines -> 15 nulos imputados
    MFN Dutiable Tariff Lines Share (%) -> 15 nulos imputados
    MFN Duty Free Tariff Lines Share (%) -> 15 nulos impu

In [20]:
# REGLA 8 - Flag de exportaciones en cero
df_clean['flag_export_cero'] = (df_clean['Export (US$ Thousand)'] == 0).astype(int)
n_export_cero = int(df_clean['flag_export_cero'].sum())
print('REGLA 8 - flag_export_cero creado. Filas afectadas: {}'.format(n_export_cero))

REGLA 8 - flag_export_cero creado. Filas afectadas: 20


In [21]:
# REGLA 9 - Flag de valores extremos en MFN AVE
df_clean['flag_mfn_ave_extremo'] = (df_clean['MFN AVE Tariff Lines Share (%)'] > 100).astype(int)
n_mfn_ave_extremo = int(df_clean['flag_mfn_ave_extremo'].sum())
print('REGLA 9 - flag_mfn_ave_extremo creado. Filas afectadas: {}'.format(n_mfn_ave_extremo))

REGLA 9 - flag_mfn_ave_extremo creado. Filas afectadas: 3632


In [22]:
# REGLA 10 - Crear variables derivadas
df_clean['Export (US$ Million)'] = (df_clean['Export (US$ Thousand)'] / 1000).round(3)
df_clean['Import (US$ Million)'] = (df_clean['Import (US$ Thousand)'] / 1000).round(3)
df_clean['Trade Balance (US$ Thousand)'] = df_clean['Export (US$ Thousand)'] - df_clean['Import (US$ Thousand)']
df_clean['Trade Balance (US$ Million)'] = (df_clean['Trade Balance (US$ Thousand)'] / 1000).round(3)
df_clean['Total Trade (US$ Thousand)'] = df_clean['Export (US$ Thousand)'] + df_clean['Import (US$ Thousand)']
df_clean['Total Trade (US$ Million)'] = (df_clean['Total Trade (US$ Thousand)'] / 1000).round(3)
df_clean['Trade Status'] = df_clean['Trade Balance (US$ Thousand)'].apply(
    lambda x: 'Superavit' if x > 0 else ('Deficit' if x < 0 else 'Equilibrio')
)

columnas_derivadas = [
    'Export (US$ Million)',
    'Import (US$ Million)',
    'Trade Balance (US$ Thousand)',
    'Trade Balance (US$ Million)',
    'Total Trade (US$ Thousand)',
    'Total Trade (US$ Million)',
    'Trade Status',
]

print('REGLA 10 - Variables derivadas creadas: {}'.format(columnas_derivadas))

REGLA 10 - Variables derivadas creadas: ['Export (US$ Million)', 'Import (US$ Million)', 'Trade Balance (US$ Thousand)', 'Trade Balance (US$ Million)', 'Total Trade (US$ Thousand)', 'Total Trade (US$ Million)', 'Trade Status']


In [23]:
# REGLA 11 - Correccion de tipos y ordenamiento final
df_clean['Year'] = df_clean['Year'].astype(int)
df_clean = df_clean.sort_values(['Partner Name', 'Year']).reset_index(drop=True)
n_filas_final = len(df_clean)
print('REGLA 11 - Dataset ordenado y tipos corregidos. Filas finales: {:,}'.format(n_filas_final))

REGLA 11 - Dataset ordenado y tipos corregidos. Filas finales: 7,783


## 7. Bitacora de transformaciones

In [24]:
bitacora = pd.DataFrame([
    [
        'B-001',
        'Columnas constantes o redundantes',
        ', '.join(COLS_BASURA),
        'Columnas con valor unico o redundantes con otra variable',
        'Eliminar columnas',
        'Columnas presentes con 0 variabilidad',
        'Columnas eliminadas del dataset limpio',
        'Alta',
    ],
    [
        'B-002',
        'Agregados regionales',
        'Partner Name',
        'Filas agregadas (World y regiones) mezcladas con paises',
        'Separar y excluir del dataset analitico',
        '{} filas agregadas'.format(n_agregados),
        'Filas agregadas removidas del dataset limpio',
        'Alta',
    ],
    [
        'B-003',
        'Entidades con cobertura insuficiente',
        'Partner Name',
        'Territorios con menos de 10 anios de datos (no extintos)',
        'Excluir filas con cobertura insuficiente',
        '{} filas antes'.format(n_antes_cobertura),
        '{} filas despues'.format(n_despues_cobertura),
        'Alta',
    ],
    [
        'B-004',
        'Clasificacion historica',
        'entity_status',
        'Distinguir paises activos vs extintos para storytelling',
        'Crear columna derivada',
        'No existia',
        'entity_status con valores Activo/Extinto',
        'Media',
    ],
    [
        'B-005',
        'World Growth con nulos',
        'World Growth (%)',
        'Valores faltantes por anio',
        'Completar por valor del anio (broadcast)',
        '{} nulos'.format(nulos_antes_wg),
        '{} nulos'.format(nulos_despues_wg),
        'Media',
    ],
    [
        'B-006',
        'Nulos en tarifas AHS/MFN',
        'Columnas AHS/MFN',
        'Nulos puntuales (~0.2%) en variables arancelarias',
        'Imputar mediana por anio',
        '{} nulos'.format(total_imputados_tarifas),
        'Nulos imputados con mediana por anio',
        'Media',
    ],
    [
        'B-007',
        'Exportaciones en cero',
        'Export (US$ Thousand)',
        'Posibles faltantes codificados como cero',
        'Crear flag_export_cero',
        '{} filas con Export=0'.format(n_export_cero),
        'flag_export_cero = 1 para filas afectadas',
        'Media',
    ],
    [
        'B-008',
        'MFN AVE extremo',
        'MFN AVE Tariff Lines Share (%)',
        'Valores > 100 por definicion AVE',
        'Crear flag_mfn_ave_extremo',
        '{} filas con AVE > 100'.format(n_mfn_ave_extremo),
        'flag_mfn_ave_extremo = 1 para filas afectadas',
        'Media',
    ],
    [
        'B-009',
        'Variables derivadas',
        ', '.join(columnas_derivadas),
        'Metricas derivadas para analisis y Tableau',
        'Crear columnas derivadas',
        'No existian',
        'Nuevas columnas analiticas',
        'Baja',
    ],
    [
        'B-010',
        'Correccion de tipos y ordenamiento',
        'Year, ordenamiento',
        'Year debe ser int y el orden facilita series temporales',
        'Convertir Year y ordenar por Partner Name y Year',
        'Orden original',
        'Ordenado y tipos consistentes',
        'Baja',
    ],
], columns=[
    'ID',
    'Tipo de problema',
    'Campo(s)',
    'Descripcion',
    'Accion aplicada',
    'Dato original',
    'Dato transformado',
    'Impacto',
])

print('Bitacora de transformaciones: {} decisiones registradas'.format(len(bitacora)))
bitacora

bitacora.to_csv(RUTA_BITACORA, index=False)

Bitacora de transformaciones: 10 decisiones registradas


**Observacion sobre la bitácora:** aquí no solo se registra qué se cambió, sino por qué se cambió y qué impacto tuvo. Eso es importante porque el entregable no pide solo limpiar, sino demostrar trazabilidad.
En términos académicos, esta sección justifica que la limpieza no fue arbitraria: cada regla responde a un hallazgo previo y deja evidencia de la transformación aplicada.

## 8. Dataset limpio - validacion y exportacion

In [25]:
print('VALIDACION DEL DATASET LIMPIO')
print('=' * 60)
print('  Filas           : {:,}'.format(df_clean.shape[0]))
print('  Columnas        : {}'.format(df_clean.shape[1]))
print('  Duplicados      : {}'.format(df_clean.duplicated(subset=['Partner Name','Year']).sum()))
print('  Paises unicos   : {}'.format(df_clean['Partner Name'].nunique()))
print('  Rango temporal  : {} - {}'.format(df_clean['Year'].min(), df_clean['Year'].max()))
print()
cols_criticas = [
    'Partner Name', 'Year',
    'Export (US$ Million)', 'Import (US$ Million)',
    'Trade Balance (US$ Million)', 'Total Trade (US$ Million)',
    'Trade Status', 'flag_export_cero',
]
nulos_criticos = df_clean[cols_criticas].isnull().sum()
print('Nulos en variables criticas:')
print(nulos_criticos.to_string())

VALIDACION DEL DATASET LIMPIO
  Filas           : 7,783
  Columnas        : 38
  Duplicados      : 0
  Paises unicos   : 252
  Rango temporal  : 1988 - 2021

Nulos en variables criticas:
Partner Name                   0
Year                           0
Export (US$ Million)           0
Import (US$ Million)           0
Trade Balance (US$ Million)    0
Total Trade (US$ Million)      0
Trade Status                   0
flag_export_cero               0


In [26]:
print('VERIFICACION DE COMPATIBILIDAD CON TABLEAU')
print('=' * 60)
problemas_tableau = []
for col in df_clean.columns:
    if df_clean[col].dtype == object:
        tipo_tableau = 'String'
    elif df_clean[col].dtype == 'int64':
        tipo_tableau = 'Integer'
    elif df_clean[col].dtype == 'float64':
        tipo_tableau = 'Float'
    else:
        tipo_tableau = 'REVISAR'
        problemas_tableau.append(col)
    print('  {} -> {}'.format(col, tipo_tableau))

if problemas_tableau:
    print('Columnas con tipo ambiguo: {}'.format(problemas_tableau))
else:
    print('Tipos compatibles con Tableau: OK')

VERIFICACION DE COMPATIBILIDAD CON TABLEAU
  Partner Name -> String
  Year -> Integer
  Export (US$ Thousand) -> Float
  Import (US$ Thousand) -> Float
  World Growth (%) -> Float
  AHS Simple Average (%) -> Float
  AHS Weighted Average (%) -> Float
  AHS Total Tariff Lines -> Float
  AHS Dutiable Tariff Lines Share (%) -> Float
  AHS Duty Free Tariff Lines Share (%) -> Float
  AHS Specific Tariff Lines Share (%) -> Float
  AHS AVE Tariff Lines Share (%) -> Float
  AHS MaxRate (%) -> Float
  AHS MinRate (%) -> Float
  AHS SpecificDuty Imports (US$ Thousand) -> Float
  AHS Dutiable Imports (US$ Thousand) -> Float
  AHS Duty Free Imports (US$ Thousand) -> Float
  MFN Simple Average (%) -> Float
  MFN Weighted Average (%) -> Float
  MFN Total Tariff Lines -> Float
  MFN Dutiable Tariff Lines Share (%) -> Float
  MFN Duty Free Tariff Lines Share (%) -> Float
  MFN Specific Tariff Lines Share (%) -> Float
  MFN AVE Tariff Lines Share (%) -> Float
  MFN MaxRate (%) -> Float
  MFN SpecificDut

In [27]:
df_clean.to_csv(RUTA_LIMPIO, index=False, encoding='utf-8-sig')
print('Dataset limpio exportado: {}'.format(RUTA_LIMPIO))
print('  Filas  : {:,}'.format(df_clean.shape[0]))
print('  Cols   : {}'.format(df_clean.shape[1]))

if len(df_agregados) > 0:
    df_agregados.to_csv(RUTA_AGREGADOS, index=False, encoding='utf-8-sig')
    print('Dataset de agregados exportado: {}'.format(RUTA_AGREGADOS))
    print('  Filas  : {:,}'.format(df_agregados.shape[0]))

Dataset limpio exportado: ../data/processed/dataset_limpio_entrega2_consolidado.csv
  Filas  : 7,783
  Cols   : 38
Dataset de agregados exportado: ../data/processed/dataset_agregados_referencia.csv
  Filas  : 272


**Sobre la exportación auxiliar:** `dataset_agregados_referencia.csv` no compite con el dataset limpio; lo acompaña. Guarda las filas de `World` y regiones para que no se pierda la referencia original y, si hace falta, se puedan revisar por separado en una etapa de auditoría o documentación.
Para el análisis principal y para Tableau, el archivo que debe usarse es el dataset limpio. El de agregados solo existe para trazabilidad y referencia.

## 9. Modelado de datos

El dataset limpio es una tabla plana (flat table) con granularidad 1 fila = 1 pais por anio.

**Clave primaria:** (Partner Name, Year)

**Dimensiones:** Partner Name, Year, entity_status, Trade Status, flags

**Metricas:** Export/Import, Trade Balance, Total Trade, metricas arancelarias AHS/MFN

**Posibles joins externos:**
- Partner Name -> ISO 3166 (normalizar nombres, regiones)
- Year -> indicadores macro (WDI)

In [28]:
print('RESUMEN DEL MODELO DE DATOS')
print('=' * 60)
print('  Tipo de modelo   : Tabla plana (flat table)')
print('  Granularidad     : Pais x Anio')
print('  Clave primaria   : (Partner Name, Year)')
print('  Total variables  : {}'.format(df_clean.shape[1]))
print()

grupos = {
    'Identificadores / Dimensiones': [
        col for col in ['Partner Name', 'Year', 'entity_status', 'Trade Status', 'flag_export_cero', 'flag_mfn_ave_extremo']
        if col in df_clean.columns
    ],
    'Metricas comerciales (miles USD)': [
        col for col in ['Export (US$ Thousand)', 'Import (US$ Thousand)', 'Trade Balance (US$ Thousand)', 'Total Trade (US$ Thousand)']
        if col in df_clean.columns
    ],
    'Metricas comerciales (millones USD)': [
        col for col in ['Export (US$ Million)', 'Import (US$ Million)', 'Trade Balance (US$ Million)', 'Total Trade (US$ Million)']
        if col in df_clean.columns
    ],
    'Metricas de crecimiento': [
        col for col in ['World Growth (%)'] if col in df_clean.columns
    ],
    'Metricas arancelarias AHS': [c for c in df_clean.columns if c.startswith('AHS')],
    'Metricas arancelarias MFN': [c for c in df_clean.columns if c.startswith('MFN')],
}

for grupo, cols in grupos.items():
    print('  {}: {} variables'.format(grupo, len(cols)))

print()
print('Dataset listo para conectar a Tableau.')

RESUMEN DEL MODELO DE DATOS
  Tipo de modelo   : Tabla plana (flat table)
  Granularidad     : Pais x Anio
  Clave primaria   : (Partner Name, Year)
  Total variables  : 38

  Identificadores / Dimensiones: 6 variables
  Metricas comerciales (miles USD): 4 variables
  Metricas comerciales (millones USD): 4 variables
  Metricas de crecimiento: 1 variables
  Metricas arancelarias AHS: 12 variables
  Metricas arancelarias MFN: 11 variables

Dataset listo para conectar a Tableau.
